In [17]:
print('x')

x


In [18]:
print('x')

x


In [21]:

import kagglehub

# Download latest version
path = kagglehub.dataset_download("ajverse/customer-support-tickets-crm-dataset")

print("Path to dataset files:", path)


import pandas as pd
df = pd.read_csv(f"{path}/customer_support_tickets.csv")
df.head()




Path to dataset files: /Users/suyashbiranje/.cache/kagglehub/datasets/ajverse/customer-support-tickets-crm-dataset/versions/2


,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,Anya Sharma,5
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,David Kim,5


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import re

x = df["Ticket_Description"].fillna("")
y = df["Priority_Level"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

def clean_reviews(i):
    i = str(i).lower()
    i = re.sub(r"<[^>]+>", "", i)
    i = re.sub(r"https?://\S+|www\.\S+", "", i)
    i = re.sub(r"[^a-zA-Z\s']", "", i)
    return i.strip()

x_train = x_train.apply(clean_reviews)
x_test = x_test.apply(clean_reviews)

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=5000
)
x_train_tfidf = tfidf_vectorizer.fit_transform(x_train)
x_test_tfidf = tfidf_vectorizer.transform(x_test)

label_mapping = {label: idx for idx, label in enumerate(sorted(y_train.unique()))}
y_train_num = y_train.map(label_mapping)
y_test_num = y_test.map(label_mapping)

lr = LogisticRegression(max_iter=1000)
lr.fit(x_train_tfidf, y_train_num)
pred = lr.predict(x_test_tfidf)

print(classification_report(
    y_test_num,
    pred,
    target_names=[k for k, v in sorted(label_mapping.items(), key=lambda kv: kv[1])]
))

              precision    recall  f1-score   support

    Critical       0.69      0.48      0.57       260
        High       0.30      0.15      0.20       683
         Low       0.49      0.62      0.55      1543
      Medium       0.39      0.40      0.40      1514

    accuracy                           0.45      4000
   macro avg       0.47      0.41      0.43      4000
weighted avg       0.43      0.45      0.43      4000

